# Notebook 1 – Setup & Data Preparation

**Purpose:** Install all required packages, verify the AWS environment, configure your S3 bucket, and make sure the banana-ripeness dataset is available in S3 before training.

**Run this notebook FIRST before running any other notebook.**

---
### What this notebook does
1. Installs Python dependencies
2. Confirms that AWS credentials and SageMaker role are working
3. Creates (or reuses) an S3 bucket and a `config.json` that stores the bucket name
4. Walks you through uploading the dataset to S3 (two options: automatic Roboflow download **or** manual zip upload)
5. Verifies the dataset structure in S3

## Step 1 – Install Required Packages

> This may take 2–3 minutes. You only need to run it once per JupyterLab instance.

In [ ]:
import sys

packages = [
    "torch",
    "torchvision",
    "boto3",
    "sagemaker",
    "Pillow",
    "matplotlib",
    "scikit-learn",
    "seaborn",
    "tqdm",
]

try:
    import subprocess
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + packages,
        capture_output=True, text=True, check=True
    )
    print("✅ All packages installed successfully.")
except subprocess.CalledProcessError as e:
    print("❌ Package installation failed:")
    print(e.stderr)
    raise

## Step 2 – Verify AWS Environment

In [ ]:
import boto3
import sagemaker
import json
import os

print("=" * 50)
print("AWS Environment Check")
print("=" * 50)

try:
    # Retrieve the SageMaker execution role
    role = sagemaker.get_execution_role()
    print(f"✅ SageMaker Role  : {role}")
except Exception as e:
    print(f"⚠️  Could not auto-detect SageMaker role: {e}")
    print("   If you are running outside SageMaker, make sure your AWS credentials")
    print("   are configured (run 'aws configure' in a terminal).")
    role = None

try:
    session = sagemaker.Session()
    region  = session.boto_region_name
    account = boto3.client("sts").get_caller_identity()["Account"]
    print(f"✅ AWS Region      : {region}")
    print(f"✅ AWS Account ID  : {account}")
except Exception as e:
    print(f"❌ AWS session error: {e}")
    raise

# Show PyTorch and GPU info
try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"✅ PyTorch version : {torch.__version__}")
    print(f"✅ CUDA available  : {cuda_available}")
    if cuda_available:
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("   ⚠️  No GPU found – training will be slow on CPU.")
        print("   Consider switching to a GPU-based instance (e.g. ml.g4dn.xlarge).")
except ImportError:
    print("❌ PyTorch is not installed – re-run Step 1.")
    raise

## Step 3 – Configure S3 Bucket

The cell below automatically uses the **SageMaker default bucket**.

If you want to use your own bucket, change `CUSTOM_BUCKET` to its name (it must already exist).

In [ ]:
# ── ⚙️  OPTIONAL: Set your own bucket name here, or leave as None ──────────────
CUSTOM_BUCKET = None   # e.g. "my-existing-bucket"
# ────────────────────────────────────────────────────────────────────────────────

S3_DATA_PREFIX  = "banana-ripeness-dataset-original"
S3_MODEL_PREFIX = "banana-ripeness-models"

try:
    session = sagemaker.Session()
    if CUSTOM_BUCKET:
        S3_BUCKET = CUSTOM_BUCKET
        # Verify the bucket is accessible
        boto3.client("s3").head_bucket(Bucket=S3_BUCKET)
        print(f"✅ Using custom bucket : s3://{S3_BUCKET}")
    else:
        S3_BUCKET = session.default_bucket()
        print(f"✅ Using SageMaker default bucket : s3://{S3_BUCKET}")
except Exception as e:
    print(f"❌ Could not access bucket: {e}")
    raise

# Persist config so other notebooks can read it
CONFIG_FILE = "config.json"
config = {
    "S3_BUCKET":      S3_BUCKET,
    "S3_DATA_PREFIX":  S3_DATA_PREFIX,
    "S3_MODEL_PREFIX": S3_MODEL_PREFIX,
    "NUM_CLASSES":     6,
    "CLASS_NAMES":     ["freshripe", "freshunripe", "overripe", "ripe", "rotten", "unripe"],
}
try:
    with open(CONFIG_FILE, "w") as f:
        json.dump(config, f, indent=2)
    print(f"✅ Config saved to {CONFIG_FILE}")
    print(json.dumps(config, indent=2))
except OSError as e:
    print(f"❌ Could not write config file: {e}")
    raise

## Step 4 – Upload Dataset to S3

### Option A – You already have the dataset zip on this machine
1. Download the Banana Ripeness Classification dataset from [Roboflow Universe](https://universe.roboflow.com/musa-acuminata/banana-ripeness-classification) (free account required) — select **"Original Images"** format and download as a **zip**.
2. Upload the zip file to the JupyterLab file browser (drag-and-drop).
3. Set `LOCAL_ZIP_PATH` in the cell below to point to the uploaded zip, then run the cell.

### Option B – Dataset is already in S3
Skip to **Step 5** and just verify the structure.

---
> **Expected folder structure inside the zip (or inside the S3 prefix after upload):**
>
> ```
> banana-ripeness-dataset-original/
>     train/
>         freshripe/   freshunripe/   overripe/   ripe/   rotten/   unripe/
>     valid/
>         freshripe/   freshunripe/   overripe/   ripe/   rotten/   unripe/
>     test/
>         freshripe/   freshunripe/   overripe/   ripe/   rotten/   unripe/
> ```

In [ ]:
import zipfile
import shutil
import pathlib

# ── ⚙️  Set the path to your dataset zip file ───────────────────────────────────
LOCAL_ZIP_PATH = "Banana Ripeness Classification.v1-original-images.zip"
# ────────────────────────────────────────────────────────────────────────────────

EXTRACT_DIR = "/tmp/banana_dataset"

if not os.path.exists(LOCAL_ZIP_PATH):
    print(f"⚠️  Zip file not found at: {LOCAL_ZIP_PATH}")
    print("   Please upload the zip file and update LOCAL_ZIP_PATH, then re-run.")
    print("   If your dataset is already in S3, skip to Step 5.")
else:
    try:
        # Extract
        print(f"📦 Extracting {LOCAL_ZIP_PATH} → {EXTRACT_DIR} ...")
        if os.path.exists(EXTRACT_DIR):
            shutil.rmtree(EXTRACT_DIR)
        os.makedirs(EXTRACT_DIR, exist_ok=True)

        with zipfile.ZipFile(LOCAL_ZIP_PATH, "r") as zf:
            zf.extractall(EXTRACT_DIR)
        print("✅ Extraction complete.")

        # Show top-level structure
        print("\nExtracted contents:")
        for item in sorted(pathlib.Path(EXTRACT_DIR).rglob("*"))[:30]:
            print(" ", item.relative_to(EXTRACT_DIR))

        # Upload to S3 using SageMaker SDK (handles multi-part, retries automatically)
        print(f"\n📤 Uploading dataset to s3://{S3_BUCKET}/{S3_DATA_PREFIX} ...")
        session = sagemaker.Session()
        s3_uri = session.upload_data(
            path=EXTRACT_DIR,
            bucket=S3_BUCKET,
            key_prefix=S3_DATA_PREFIX,
        )
        print(f"✅ Dataset uploaded to: {s3_uri}")

    except zipfile.BadZipFile:
        print(f"❌ The file '{LOCAL_ZIP_PATH}' is not a valid zip archive.")
        raise
    except Exception as e:
        print(f"❌ Error during extraction / upload: {e}")
        raise

## Step 5 – Verify Dataset in S3

This cell lists what is inside your S3 bucket under the data prefix and checks that the expected splits (train / valid / test) exist.

In [ ]:
with open(CONFIG_FILE) as f:
    config = json.load(f)

S3_BUCKET      = config["S3_BUCKET"]
S3_DATA_PREFIX = config["S3_DATA_PREFIX"]
CLASS_NAMES    = config["CLASS_NAMES"]

s3 = boto3.client("s3")

print(f"Checking s3://{S3_BUCKET}/{S3_DATA_PREFIX}/\n")

REQUIRED_SPLITS = ["train", "valid", "test"]
found_splits = set()

try:
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_DATA_PREFIX + "/", Delimiter="/")

    top_level = []
    for page in pages:
        for prefix in page.get("CommonPrefixes", []):
            top_level.append(prefix["Prefix"])

    print("Top-level folders found:")
    for p in top_level:
        folder = p.rstrip("/").split("/")[-1]
        print(f"  📁 {folder}")
        if folder in REQUIRED_SPLITS:
            found_splits.add(folder)

    print()
    all_ok = True
    for split in REQUIRED_SPLITS:
        if split in found_splits:
            # Count images in this split
            resp = s3.list_objects_v2(
                Bucket=S3_BUCKET,
                Prefix=f"{S3_DATA_PREFIX}/{split}/",
            )
            count = resp.get("KeyCount", 0)
            print(f"  ✅ {split:6s} – {count} objects")
        else:
            print(f"  ❌ '{split}' folder NOT FOUND in S3.")
            all_ok = False

    print()
    if all_ok:
        print("🎉 Dataset looks good! You can now run Notebook 2 (02_train_models.ipynb).")
    else:
        print("⚠️  Some folders are missing. Please re-upload the dataset (Step 4).")

except s3.exceptions.NoSuchBucket:
    print(f"❌ Bucket '{S3_BUCKET}' does not exist or you do not have access.")
    raise
except Exception as e:
    print(f"❌ Error while listing S3 objects: {e}")
    raise

---
## ✅ Setup Complete!

Your environment is ready. Next step → **open and run `02_train_models.ipynb`**.